# 01. 표본 정의와 건수 패널 구축
> 계획서: `계획/P1_표본정의_패널구축.md` · 체크리스트를 먼저 읽고 시작하세요.

**이 노트북이 하는 일**
1. 원본에서 건수 8개(창구·인터넷·스마트·폰·ATM·자동이체·외환 수출·외환 수입)와 입금·출금·잔액·속성 열만 읽어 법인 × 36개월 패널을 만든다.
2. 기대값(행 365,988 / 법인 15,473 / P36 3,372 / P35 1,412)과 비교한다. 하나라도 어긋나면 멈춘다.
3. 표본을 정의한다. **P36** = 36개월 완전관측, **P35+** = 35개월 이상(P36 + 정확히 35개월). 35개월 기업의 빈 달이 어디에 있는지 본다.
4. 빈 달 처리 규칙을 정의한다. **M0** = 채우지 않고 관측된 달로 계산, **M1** = 채움. 채움 규칙이 얼마나 정확한지 36개월 기업으로 시험한다.

**규칙**: 화면 출력은 집계값뿐 · 5 미만 칸은 `<5` · 법인ID는 어디에도 남기지 않음 · 결과를 공유하기 전에 직접 검토 · 노트북을 커밋하기 전에 `tools/strip_outputs.py` 실행.

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── 설정: 여기만 고치세요 ────────────────────────────────────────────
WORK_DIR = os.environ.get("CNT_WORK_DIR") or None   # 기본값: 레포의 data/work_cnt (.gitignore 대상)
# ────────────────────────────────────────────────────────────────────

def _find_lib():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "code", base / "건수분석" / "code", base / "yangdaecheon" / "건수분석" / "code"):
            if (cand / "cntlib.py").exists():
                return cand
    raise SystemExit("cntlib.py 를 찾지 못했습니다. 노트북을 yangdaecheon/건수분석/notebooks 에서 여세요.")

sys.path.insert(0, str(_find_lib()))
import cntlib as cl
cl.setup_korean_font()
WORK = cl.resolve_work_dir(WORK_DIR)
OUT = WORK / "output"
print("산출물 폴더:", WORK)

# 원본 파일 경로: 레포 밖에 두세요. 경로를 코드 밖으로 내보내지 마세요.
DATA_PATH = os.environ.get("CNT_DATA_PATH", r"여기에_원본_경로.xlsx")
SHEET = os.environ.get("CNT_SHEET", "all")          # 시트가 하나면 "0"
EXPECT = None if os.environ.get("CNT_NO_EXPECT") else dict(rows=365988, firms=15473, p36=3372, p35=1412)

## 1. 원본 읽기와 패널 구축 (엑셀이면 몇 분 걸립니다)

In [ ]:
t0 = time.time()
raw = cl.read_raw(DATA_PATH, SHEET)
print(f"원본 읽기 {time.time() - t0:.0f}초, 행 {len(raw):,}")
panel, rep = cl.build_panel(raw)
del raw
if panel.synthetic:
    print(cl.SYNTHETIC_NOTICE)
masks = cl.sample_masks(panel)
nobs = masks["nobs"]
print(f"패널 구축 완료: 법인 {panel.n:,} x {cl.T}개월")

## 2. 검증표 (전부 PASS 여야 다음으로 진행)

In [ ]:
rows = []
def chk(name, expect, actual):
    ok = True if expect is None else (expect == actual)
    f = lambda v: f"{v:,}" if isinstance(v, (int, np.integer)) else str(v)
    rows.append((name, "-" if expect is None else f(expect), f(actual), "PASS" if ok else "FAIL"))

chk("행 수", EXPECT and EXPECT["rows"], rep["rows"])
chk("고유 법인 수", EXPECT and EXPECT["firms"], rep["firms"])
chk("(법인, 월) 중복", 0, rep["dup_pairs"])
chk("관측된 월 수", 36, rep["months_present"])
chk("P36 (36개월 완전관측) 법인 수", EXPECT and EXPECT["p36"], int(masks["P36"].sum()))
chk("P35 (정확히 35개월) 법인 수", EXPECT and EXPECT["p35"], int(masks["P35"].sum()))
chk("거래건수 구간 라벨 수", 10, len(rep["band_labels"]))
for c in cl.AMT_COLS:
    chk(f"금액 변환 실패: {c}", 0, rep["parse_fail"][c])
    chk(f"음수 금액: {c}", 0, rep["negatives"][c])
tab = pd.DataFrame(rows, columns=["점검", "기대", "실제", "결과"])
cl.show_table(tab, "검증표", OUT / "nb01", "validation")
assert (tab["결과"] == "PASS").all(), "검증 실패: 다음 단계로 가지 말고 FAIL 행을 확인하세요."
cl.show_table(pd.DataFrame({"코드": range(10), "구간 라벨(코드북)": panel.band_labels}), "거래건수 구간 코드", OUT / "nb01", "band_codes")

## 3. 표본 정의: P36 vs P35+
- **P36**: 36개월 모두 관측 (기존 분석 대상)
- **P35**: 정확히 35개월 관측 (한 달이 비어 있음)
- **P35+**: 35개월 이상 = P36 + P35 (이번 팀 회의에서 추가하기로 한 표본)

In [ ]:
dist = pd.Series(nobs).value_counts().reindex(range(30, 37), fill_value=0)
cl.show_table(pd.DataFrame({"관측 개월 수": dist.index, "법인 수": [cl.sup(x) for x in dist.values]}),
              "관측 개월 수 (30~36개월)", OUT / "nb01", "obs_months_30_36")
n36, n35, n35p = int(masks["P36"].sum()), int(masks["P35"].sum()), int(masks["P35p"].sum())
samp = pd.DataFrame({"표본": ["P36 (36개월 완전관측)", "P35 (정확히 35개월)", "P35+ (35개월 이상)"],
                     "법인 수": [cl.sup(n36), cl.sup(n35), cl.sup(n35p)],
                     "전체 대비(%)": [round(v / panel.n * 100, 1) for v in (n36, n35, n35p)],
                     "P36 대비 표본 증가(%)": [0.0, "-", round((n35p / n36 - 1) * 100, 1)]})
cl.show_table(samp, "표본 정의", OUT / "nb01", "samples")
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.bar(dist.index, dist.values, color=["#2a78d6" if k >= 35 else "#c9c8c1" for k in dist.index])
ax.set_xlabel("관측 개월 수"); ax.set_ylabel("법인 수"); ax.set_title("관측 개월 수 30~36개월 (파랑: P35+)")
plt.tight_layout(); plt.show()

## 4. 35개월 기업의 빈 달은 어디에 있나
빈 달이 **첫 달(2023.01)이나 마지막 달(2025.12)** 이면 '채운다'기보다 '창이 한 달 짧아진다'에 가깝고, **중간**이면 앞뒤 관측값으로 보간할 수 있습니다.

In [ ]:
pos = cl.missing_position(panel, masks["P35"])
cnt_pos = np.bincount(pos, minlength=cl.T)
kind = np.where(pos == 0, "첫 달 (2023.01)", np.where(pos == cl.T - 1, "마지막 달 (2025.12)", "중간"))
ks = pd.Series(kind).value_counts().reindex(["첫 달 (2023.01)", "중간", "마지막 달 (2025.12)"], fill_value=0)
cl.show_table(pd.DataFrame({"빈 달 위치": ks.index, "법인 수": [cl.sup(x) for x in ks.values],
                            "비율(%)": [round(x / max(len(pos), 1) * 100, 1) for x in ks.values]}),
              "35개월 기업의 빈 달 위치", OUT / "nb01", "p35_missing_kind")
cl.show_table(pd.DataFrame({"기준년월": panel.months, "빈 달이 이 달인 기업 수": [cl.sup(x) for x in cnt_pos]}),
              "빈 달이 몰리는 달 (월별)", OUT / "nb01", "p35_missing_by_month")
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar(range(cl.T), cnt_pos, color="#2a78d6")
ax.set_xticks(range(0, cl.T, 3)); ax.set_xticklabels([str(m) for m in panel.months[::3]], rotation=45)
ax.set_ylabel("법인 수"); ax.set_title("35개월 기업: 빈 달이 있는 달")
plt.tight_layout(); plt.show()

## 5. 빈 달 처리 규칙 정의와 단위 점검
- **M0 (기본)**: 채우지 않는다. 월 평균 같은 계산은 *관측된 달만으로* 한다 (창 안에서 빈 달이 1개 이하일 때만).
- **M1 (민감도)**: 같은 기업의 관측값으로 채운다. 중간 공백은 **선형 보간 후 반올림**, 맨 앞·뒤 공백은 **가장 가까운 관측값**.

In [ ]:
toy = np.array([[2, np.nan, 4, 4, np.nan, 6]], dtype=float)
assert cl.fill_interp(toy).tolist() == [[2, 3, 4, 4, 5, 6]], cl.fill_interp(toy)
toy2 = np.array([[np.nan, 3, 3, 5, np.nan]], dtype=float)
assert cl.fill_interp(toy2).tolist() == [[3, 3, 3, 5, 5]], cl.fill_interp(toy2)
toy3 = np.array([[1, np.nan, 2]], dtype=float)        # 1.5 는 반올림해서 2
assert cl.fill_interp(toy3).tolist() == [[1, 2, 2]]
print("M1 채움 규칙 단위 점검 통과")

**채움 규칙의 정확도 시험**: 36개월 기업은 정답을 알고 있으므로, 35개월 기업의 빈 달 위치 분포(위 4번)를 그대로 따라 36개월 기업의 한 달을 가렸다가 M1으로 채워 보고 정답과 비교합니다.
정확도가 낮으면 M1은 민감도 분석에만 쓰고 기본은 M0로 갑니다.

In [ ]:
rng = np.random.default_rng(20261001)
idx36 = np.flatnonzero(masks["P36"])
probs = np.bincount(pos, minlength=cl.T) / max(len(pos), 1)
hole = rng.choice(cl.T, size=len(idx36), p=probs)
ar = np.arange(len(idx36))
hk = np.where(hole == 0, "첫 달", np.where(hole == cl.T - 1, "마지막 달", "중간"))
rows, all_err, all_kind = [], [], []
for c in cl.CNT_COLS:
    true = panel.cnt[c][idx36].astype(float)
    masked = true.copy()
    masked[ar, hole] = np.nan
    d = cl.fill_interp(masked)[ar, hole] - true[ar, hole]
    all_err.append(d); all_kind.append(hk)
    rows.append({"변수": cl.SHORT[c], "정확히 일치(%)": round((d == 0).mean() * 100, 1),
                 "±1 코드 이내(%)": round((np.abs(d) <= 1).mean() * 100, 1), "평균 오차(코드)": round(d.mean(), 3)})
cl.show_table(pd.DataFrame(rows), "M1 채움 정확도 (변수별)", OUT / "nb01", "fill_accuracy_by_var")
E, K = np.concatenate(all_err), np.concatenate(all_kind)
rows = [{"빈 달 위치": k, "정확히 일치(%)": round((E[K == k] == 0).mean() * 100, 1),
         "±1 코드 이내(%)": round((np.abs(E[K == k]) <= 1).mean() * 100, 1)} for k in ["첫 달", "중간", "마지막 달"] if (K == k).any()]
cl.show_table(pd.DataFrame(rows), "M1 채움 정확도 (빈 달 위치별, 8개 변수 합산)", OUT / "nb01", "fill_accuracy_by_pos")
print("해석 기준(계획서 P1): ±1 코드 이내 90% 이상이면 M1을 민감도 분석에 쓴다. 미만이면 M0만 쓴다.")

## 6. 저장 (행 단위 데이터: 공유·커밋 금지)

In [ ]:
panel.save(WORK / "interim" / "count_panel.npz")
np.savez_compressed(WORK / "interim" / "sample_masks.npz", **masks)
summary = {"P36": n36, "P35": n35, "P35p": n35p, "missing_first": int(ks.iloc[0]), "missing_mid": int(ks.iloc[1]),
           "missing_last": int(ks.iloc[2]), "fill_within1_pct": float((np.abs(E) <= 1).mean() * 100),
           "fill_exact_pct": float((E == 0).mean() * 100), "synthetic": bool(panel.synthetic)}
cl.save_json(WORK / "interim" / "nb01_summary.json", summary)
print("저장 완료:", WORK / "interim")
print(summary)

## 체크포인트 (Claude에게 알려줄 것)
공유 가능한 범위(팀 확인)에서 아래만 알려주세요.
- [ ] 검증표가 전부 PASS 인가
- [ ] P36 / P35 / P35+ 법인 수
- [ ] 35개월 기업의 빈 달 위치 비율 (첫 달 / 중간 / 마지막 달)
- [ ] M1 채움 정확도: ±1 코드 이내 비율 (전체와 위치별)

**다음**: 계획서 `P1` 체크리스트를 채우고 `P2` 계획을 확인한 뒤 `02_건수수치화.ipynb` 로 갑니다.